# Base imports

In [ ]:
library(devtools)

In [ ]:
library(tidyverse)

In [ ]:
library(here)

In [ ]:
library(powerjoin)

In [ ]:
library(qs)

In [ ]:
library(Rsubread)

In [ ]:
library(ggbeeswarm)

In [ ]:
library(Rsubread)

In [ ]:
library(GenomicRanges)

## Prepare BAM files - for target als

In [ ]:

bam_files <- tibble(
  bam_path = Sys.glob(file.path("/home/rob6090/scr_proj/tdp-43/data/ad_bam/*.sorted.bam"))
) %>%
  filter(!str_detect(bam_path, fixed("markdup"))) %>%
  mutate(
    sample_id = str_extract(bam_path, "([^/]+)\\.sorted\\.bam", group = 1)
  )

In [ ]:
head(bam_files)

## Load RepeatMasker file

In [ ]:
repeatmasker_raw <- read_csv("~/main/projects/genomics/tdp-43/data/repeatmasker_raw.csv") 

In [ ]:
repeats_saf <- repeatmasker_raw%>%
  transmute(
    GeneID = paste(rep_name, row_number(), sep = "_"),
    Chr = chromosome,
    Start = start,
    End = end,
    Strand = case_when(
      strand == "+" ~ "+",
      strand == "C" ~ "-",
      TRUE ~ "*"
    )
  )

In [14]:
1

[1] 1

In [ ]:
head(repeats_saf)

## Run featureCounts

In [ ]:

fc_results <- featureCounts(
  files = bam_files$bam_path,
  annot.ext = repeats_saf,
  isGTFAnnotationFile = FALSE,
  allowMultiOverlap = TRUE,
  countMultiMappingReads = TRUE,               # -M
  fraction = TRUE,               # --fraction
  primaryOnly = FALSE,
  isPairedEnd = TRUE,
  nthreads = 18
)

## Unique only feature counts

In [ ]:
# Run featureCounts
fc_results <- featureCounts(
  files = bam_files$bam_path,
  annot.ext = repeats_saf,
  isGTFAnnotationFile = FALSE,
  allowMultiOverlap = TRUE,
  #countMultiMappingReads = TRUE,               # -M
  #fraction = TRUE,               # --fraction
  primaryOnly = FALSE,
  isPairedEnd = TRUE,
  nthreads = 18
)

In [ ]:
qsave(
  fc_results,
  "~/tdp-43/scripts/analysis/final_dsRNA/ad_repeat_counts_raw-fraction_AND_secondary_noMETA.qs"
)

# additional tests - fractional counts

In [ ]:
type(fc_results)

In [ ]:
length(fc_results[1])

In [ ]:
# Load the DESeq2 library
library(DESeq2)

In [ ]:
# Extract the count matrix from your featureCounts results
count_matrix <- fc_results$counts

In [ ]:
head(count_matrix)

In [27]:
colnames(count_matrix)

[1] "SRR8571937.sorted.bam" "SRR8571938.sorted.bam" "SRR8571939.sorted.bam"
 [4] "SRR8571940.sorted.bam" "SRR8571941.sorted.bam" "SRR8571942.sorted.bam"
 [7] "SRR8571944.sorted.bam" "SRR8571945.sorted.bam" "SRR8571947.sorted.bam"
[10] "SRR8571948.sorted.bam" "SRR8571949.sorted.bam" "SRR8571950.sorted.bam"
[13] "SRR8571951.sorted.bam" "SRR8571952.sorted.bam"

In [ ]:
sample_info

In [ ]:
rownames(sample_info) <- colnames(count_matrix)

In [ ]:
sample_info

## FIX: Round the fractional counts to the nearest integer

In [ ]:
count_matrix <- round(count_matrix)

## Run Differential Expression Analysis

In [ ]:
# Create the DESeqDataSet object
dds <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = sample_info,
  design = ~ condition
)

In [ ]:
# Run the DESeq2 analysis
dds <- DESeq(dds)

In [ ]:
qsave(
  dds,
  file.path("dsrna_STAR-fractional_AND_secondary-noMETA_11_fam", "data", "tdp-43_path_de.qs")
)

In [ ]:
# Get the results
# The contrast specifies that we are comparing the 'TDP43_negative' group to the 'TDP43_positive' group.
res <- results(dds, contrast = c("condition", "TDP43_negative", "TDP43_positive"))

In [ ]:
res2 <- results(dds, contrast = c("condition",  "TDP43_positive", "TDP43_negative"))

In [ ]:
type(res)

In [ ]:
head(res)

In [ ]:
head(res,n=10)

In [ ]:
summary(res)

In [ ]:

res_df <- as.data.frame(res)


res_df$GeneID <- rownames(res_df)


upregulated_repeats <- subset(res_df, log2FoldChange > 0 & padj < 0.05)


merged_results <- merge(upregulated_repeats, repeats_saf, by = "GeneID")


print(head(merged_results))

In [ ]:
dim(merged_results)

In [ ]:
summary(merged_results)

In [ ]:
head(merged_results,n=10)

In [ ]:
tail(merged_results,n=10)

In [53]:
head(repeatmasker_raw)

sw_score,perc_div,perc_del,perc_ins,chromosome,start,end,left,strand,rep_name,class_family,rep_start,rep_end,rep_left,id,dummy
<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>
463,1.3,0.6,1.7,chr1,10001,10468,(248945954),+,(TAACCC)n,Simple_repeat,1,463,(0),1,NA
3770,13.3,6.3,3.5,chr1,10469,11447,(248944975),C,TAR1,Satellite/telo,(399),1006,1,2,NA
535,20.4,18.1,2.5,chr1,11505,11675,(248944747),C,L1MC,LINE/L1,(2299),5648,5452,3,NA
263,29.4,1.9,1.0,chr1,11678,11780,(248944642),C,MER5B,DNA/hAT-Charlie,(74),104,1,4,NA
309,23.0,3.8,0.0,chr1,15265,15355,(248941067),C,MIR3,SINE/MIR,(119),143,49,5,NA
18,23.2,0.0,2.0,chr1,15798,15849,(248940573),+,(TGCTCC)n,Simple_repeat,1,51,(0),6,NA


In [ ]:
repeats_saf2 <- repeatmasker_raw%>%
  transmute(
    GeneID = paste(rep_name, row_number(), sep = "_"),
    Chr = chromosome,
    Class_family = class_family,
    Start = start,
    End = end,
    Strand = case_when(
      strand == "+" ~ "+",
      strand == "C" ~ "-",
      TRUE ~ "*"
    )
  )

In [ ]:
head(repeats_saf2)

In [ ]:
dim(repeats_saf2)

In [ ]:
length(unique(repeats_saf2$GeneID))

In [ ]:
head(merged_results,n=10)

In [ ]:
repeats_saf2_uni <- repeats_saf2 %>% 
  distinct(GeneID, .keep_all = TRUE)

In [ ]:
merged_with_ann <- merged_results %>% 
  left_join(
    repeats_saf2_uni %>% 
      select(GeneID, Chr, Start, End, Strand, Class_family),
    by = "GeneID",
    relationship = "many-to-one"   # <- will warn you if GeneID isn't unique
  )

In [ ]:
glimpse(merged_with_ann)

In [ ]:
dim(merged_with_ann)

In [ ]:
head(merged_with_ann)

In [65]:
unique(merged_with_ann$Class_family)

[1] "Simple_repeat"     "rRNA"              "RNA"              
 [4] "Low_complexity"    "SINE/Alu"          "LINE/L2"          
 [7] "SINE/5S-Deu-L2"    "DNA/hAT-Tip100"    "scRNA"            
[10] "DNA/hAT-Blackjack" "DNA/hAT-Charlie"   "LINE/CR1"         
[13] "LTR/ERV1"          "LTR/ERVL"          "DNA"              
[16] "DNA/hAT-Ac"        "LTR/ERVL?"         "DNA/hAT-Tip100?"  
[19] "LTR?"              "Unknown"           "LINE/L1"          
[22] "DNA/hAT-Tag1"      "LTR/ERVK"          "DNA/TcMar-Mariner"
[25] "DNA/TcMar-Tc2"     "LINE/RTE-X"        "SINE/tRNA"        
[28] "LTR/ERV1?"         "LTR/Gypsy"         "LTR"              
[31] "LTR/Gypsy?"        "DNA/hAT"           "DNA/TcMar-Tigger" 
[34] "LINE/RTE-BovB"     "SINE/tRNA-RTE"     "DNA?"             
[37] "DNA/PiggyBac"      "SINE/MIR"          "LTR/ERVL-MaLR"    
[40] "LINE/Penelope"     "DNA/MULE-MuDR"     "Retroposon/SVA"   
[43] "DNA/hAT?"          "DNA/TcMar"

In [ ]:
write_csv(
  merged_with_ann,
  here("dsrna_STAR-fractional_AND_secondary-noMETA_11_fam", "data", "repeat_families_in_excess.csv")
)

In [ ]:
merged_with_ann[merged_with_ann$GeneID=='MIR3_1729538',]

In [ ]:
merged_with_ann[merged_with_ann$log2FoldChange>=7,]

In [ ]:
df_sorted <- merged_with_ann[order(merged_with_ann$log2FoldChange, decreasing = TRUE), ]

In [ ]:
head(df_sorted,n=10)

In [ ]:
dim(df_sorted[df_sorted$Class_family=='Simple_repeat',])

In [ ]:
dim(df_sorted[df_sorted$Class_family=='SINE/MIR',])

In [ ]:
# SINE/Alu
dim(df_sorted[df_sorted$Class_family=='SINE/Alu',])

In [ ]:
# SINE/Alu
dim(df_sorted[df_sorted$Class_family=='LINE/L2',])

In [ ]:
# SINE/Alu
dim(df_sorted[df_sorted$Class_family=='LINE/L1',])

In [ ]:
# additional tests - Unique only mapping of reads

In [ ]:
# Extract the count matrix from the featureCounts results
count_matrix <- fc_results$counts

In [ ]:
dim(count_matrix)

In [ ]:
head(count_matrix)

In [ ]:
sample_info <- data.frame(
  sample_name = colnames(count_matrix),
  condition = factor(c("TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative")) # Adjust this to your actual sample conditions
)
rownames(sample_info) <- colnames(count_matrix)

In [ ]:
# create the DESeqDataSet object
dds <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = sample_info,
  design = ~ condition
)

In [ ]:
# run the DESeq2 analysis
dds <- DESeq(dds)

In [ ]:
1

In [ ]:
qsave(
  dds,
  file.path("dsrna_STAR-fractional_AND_secondary-noMETA_11_fam", "data", "tdp-43_path_de-only_unique_mappings.qs")
)

In [ ]:
res <- results(dds, contrast = c("condition", "TDP43_negative", "TDP43_positive"))

In [ ]:
head(res,n=10)

In [ ]:
head(results(dds, contrast = c("condition",  "TDP43_positive", "TDP43_negative")),n=10)

In [ ]:
summary(res)

In [ ]:

res_df <- as.data.frame(res)


res_df$GeneID <- rownames(res_df)


upregulated_repeats <- subset(res_df, log2FoldChange > 0 & padj < 0.05)


merged_results <- merge(upregulated_repeats, repeats_saf, by = "GeneID")


print(head(merged_results))

In [ ]:
dim(merged_results)

In [ ]:
repeats_saf2 <- repeatmasker_raw%>%
  transmute(
    GeneID = paste(rep_name, row_number(), sep = "_"),
    Chr = chromosome,
    Class_family = class_family,
    Start = start,
    End = end,
    Strand = case_when(
      strand == "+" ~ "+",
      strand == "C" ~ "-",
      TRUE ~ "*"
    )
  )

In [ ]:
repeats_saf2_uni <- repeats_saf2 %>% 
  distinct(GeneID, .keep_all = TRUE)

In [ ]:
merged_with_ann <- merged_results %>% 
  left_join(
    repeats_saf2_uni %>% 
      select(GeneID, Chr, Start, End, Strand, Class_family),
    by = "GeneID",
    relationship = "many-to-one"   # <- will warn you if GeneID isn't unique
  )

In [ ]:
dim(merged_with_ann)

In [ ]:
write_csv(
  merged_with_ann,
  here("dsrna_STAR-fractional_AND_secondary-noMETA_11_fam", "data", "repeat_families_in_excess-only_unique_mappings.csv")
)

In [ ]:
merged_with_ann[merged_with_ann['log2FoldChange']>=7,]